In [5]:
import pandas as pd
import numpy as np

In [6]:
data = pd.read_csv('D:\\UITS\\8th\\NN\\Lab\\Neural-Network\\c07\\outlierAnalysis.p2\\employee_data.csv')
data

,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours
0,1,2149.014246,8.357787,82.027862,0,10.573297
1,2,1958.520710,8.560785,87.003125,2,10.139310
2,3,2194.306561,9.083051,90.026218,2,5.920432
3,4,2456.908957,9.053802,90.234903,0,12.238761
4,5,1929.753988,6.622331,87.749673,0,11.936453
...,...,...,...,...,...,...
195,196,2115.595214,7.530824,87.449918,0,11.039465
196,197,1734.842769,6.286865,88.650625,1,12.994030
197,198,492.649900,9.353872,59.558590,13,1.311234
198,199,2017.462616,7.885460,87.778534,0,16.265124


In [7]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['Monthly_Sales', 'Customer_Satisfaction', 'Task_Completion_Rate', 'Absenteeism_Days', 'Overtime_Hours']

scaler = StandardScaler()

data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

data.head()

,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours
0,1,0.545506,0.303572,-1.047948,-0.728194,0.104498
1,2,0.015983,0.518344,-0.281555,0.572153,-0.041920
2,3,0.671407,1.070905,0.184125,0.572153,-1.465285
3,4,1.401374,1.039959,0.216271,-0.728194,0.666392
4,5,-0.063981,-1.532551,-0.166556,-0.728194,0.564400


In [8]:
from scipy.spatial.distance import mahalanobis

# Select the features for Mahalanobis distance calculation
features = ['Monthly_Sales', 'Customer_Satisfaction', 'Task_Completion_Rate', 'Absenteeism_Days', 'Overtime_Hours']
X = data[features]

# Calculate the mean vector
mean_vector = np.mean(X, axis=0)

# Calculate the covariance matrix
cov_matrix = np.cov(X, rowvar=False)

# Calculate the inverse of the covariance matrix
inv_cov_matrix = np.linalg.inv(cov_matrix)

In [9]:
# Function to calculate Mahalanobis Distance

def calculate_mahalanobis(x):
    return mahalanobis(x, mean_vector, inv_cov_matrix)

In [10]:
# Calculate Mahalanobis distances for all data points
mahalanobis_distances = X.apply(calculate_mahalanobis, axis=1)

data['Mahalanobis_Distance'] = mahalanobis_distances

outlier_threshold = 3
data['Mahalanobis_Distance_Outliers'] = (data['Mahalanobis_Distance'] > outlier_threshold)

data

,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours,Mahalanobis_Distance,Mahalanobis_Distance_Outliers
0,1,0.545506,0.303572,-1.047948,-0.728194,0.104498,1.771345,False
1,2,0.015983,0.518344,-0.281555,0.572153,-0.041920,0.790501,False
2,3,0.671407,1.070905,0.184125,0.572153,-1.465285,2.092761,False
3,4,1.401374,1.039959,0.216271,-0.728194,0.666392,1.866704,False
4,5,-0.063981,-1.532551,-0.166556,-0.728194,0.564400,1.793438,False
...,...,...,...,...,...,...,...,...
195,196,0.452609,-0.571359,-0.212730,-0.728194,0.261774,1.118547,False
196,197,-0.605785,-1.887475,-0.027772,-0.078021,0.921205,2.113868,False
197,198,-4.058761,1.357436,-4.509134,7.724063,-3.020336,8.487572,True
198,199,0.179826,-0.196153,-0.162110,-0.728194,2.024806,2.226668,False


In [11]:
from sklearn.neighbors import LocalOutlierFactor

# Initialize the Local Outlier Factor model
lof = LocalOutlierFactor(n_neighbors=20, contamination='auto')
lof.fit(X)

# Get the outlier scores
data['LOF_Score'] = lof.negative_outlier_factor_

# Identify outliers based on a threshold (e.g., scores below -1.5)
threshold_lof = -1.5
data['LOF_Outliers'] = (data['LOF_Score'] < threshold_lof)

data

,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours,Mahalanobis_Distance,Mahalanobis_Distance_Outliers,LOF_Score,LOF_Outliers
0,1,0.545506,0.303572,-1.047948,-0.728194,0.104498,1.771345,False,-1.028132,False
1,2,0.015983,0.518344,-0.281555,0.572153,-0.041920,0.790501,False,-0.975992,False
2,3,0.671407,1.070905,0.184125,0.572153,-1.465285,2.092761,False,-1.148195,False
3,4,1.401374,1.039959,0.216271,-0.728194,0.666392,1.866704,False,-1.141210,False
4,5,-0.063981,-1.532551,-0.166556,-0.728194,0.564400,1.793438,False,-1.021101,False
...,...,...,...,...,...,...,...,...,...,...
195,196,0.452609,-0.571359,-0.212730,-0.728194,0.261774,1.118547,False,-0.971931,False
196,197,-0.605785,-1.887475,-0.027772,-0.078021,0.921205,2.113868,False,-1.039211,False
197,198,-4.058761,1.357436,-4.509134,7.724063,-3.020336,8.487572,True,-4.428641,True
198,199,0.179826,-0.196153,-0.162110,-0.728194,2.024806,2.226668,False,-1.143595,False


In [12]:
data['Combined_Outliers'] = data['Mahalanobis_Distance_Outliers'] & data['LOF_Outliers']

data[data['Combined_Outliers']]

,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours,Mahalanobis_Distance,Mahalanobis_Distance_Outliers,LOF_Score,LOF_Outliers,Combined_Outliers
5,6,-4.056373,-1.067193,-4.870521,3.172847,2.100596,5.892779,True,-3.538358,True,True
13,14,-4.064083,0.614207,-2.738584,5.123368,0.099954,5.449213,True,-3.198208,True,True
20,21,1.353522,2.041042,1.720495,-0.078021,1.120634,3.134824,True,-1.528536,True,True
40,41,-3.904363,-0.913460,-2.454236,3.823021,1.780134,4.878778,True,-2.953425,True,True
62,63,-0.791314,-3.504248,-0.358845,-0.078021,-0.773524,3.662635,True,-1.651594,True,True
71,72,1.413888,-1.594119,-1.592863,-0.728194,-2.001400,3.840240,True,-1.647135,True,True
125,126,1.957955,-1.359587,-1.210041,0.572153,-1.124130,3.727420,True,-1.515688,True,True
137,138,-4.032241,-0.414997,-5.496167,2.522674,-0.553376,6.028794,True,-3.562036,True,True
179,180,2.399696,-0.930153,1.061011,2.522674,2.013039,5.724780,True,-2.084320,True,True
193,194,-0.907566,1.862725,0.437075,0.572153,-2.111379,3.088399,True,-1.526913,True,True


**Isolation Forest**

In [13]:
from sklearn.ensemble import IsolationForest

isolation_forest = IsolationForest(contamination='auto', random_state=42)
isolation_forest.fit(X)

# Predict outliers (-1 for outliers, 1 for inliers)
data['Isolation_Forest_Outliers'] = isolation_forest.predict(X)

# Convert predictions to boolean (1 for inliers, -1 for outliers)
data['Isolation_Forest_Outliers'] = data['Isolation_Forest_Outliers'] == -1

data[data['Isolation_Forest_Outliers']]

,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours,Mahalanobis_Distance,Mahalanobis_Distance_Outliers,LOF_Score,LOF_Outliers,Combined_Outliers,Isolation_Forest_Outliers
5,6,-4.056373,-1.067193,-4.870521,3.172847,2.100596,5.892779,True,-3.538358,True,True,True
9,10,0.583738,2.041042,0.576310,1.222326,1.507683,3.127180,True,-1.361740,False,False,True
13,14,-4.064083,0.614207,-2.738584,5.123368,0.099954,5.449213,True,-3.198208,True,True,True
20,21,1.353522,2.041042,1.720495,-0.078021,1.120634,3.134824,True,-1.528536,True,True,True
21,22,-0.056996,-2.050546,1.532057,-0.078021,-1.630599,3.166180,True,-1.361707,False,False,True
34,35,0.817223,2.041042,1.337208,0.572153,0.157840,2.771811,False,-1.344076,False,False,True
36,37,0.305461,-2.217581,1.434453,-0.078021,-1.384768,3.126055,True,-1.338139,False,False,True
40,41,-3.904363,-0.913460,-2.454236,3.823021,1.780134,4.878778,True,-2.953425,True,True,True
62,63,-0.791314,-3.504248,-0.358845,-0.078021,-0.773524,3.662635,True,-1.651594,True,True,True
71,72,1.413888,-1.594119,-1.592863,-0.728194,-2.001400,3.840240,True,-1.647135,True,True,True


**One Class SVM**

In [14]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(nu=0.1, kernel='rbf', gamma='scale')
ocsvm.fit(X)

# Predict outliers (-1 for outliers, 1 for inliers)
data['OCSVM_Outliers'] = ocsvm.predict(X)

# Convert predictions to boolean (True for outliers)
data['OCSVM_Outliers'] = data['OCSVM_Outliers'] == -1

data[data['OCSVM_Outliers']]

,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours,Mahalanobis_Distance,Mahalanobis_Distance_Outliers,LOF_Score,LOF_Outliers,Combined_Outliers,Isolation_Forest_Outliers,OCSVM_Outliers
5,6,-4.056373,-1.067193,-4.870521,3.172847,2.100596,5.892779,True,-3.538358,True,True,True,True
13,14,-4.064083,0.614207,-2.738584,5.123368,0.099954,5.449213,True,-3.198208,True,True,True,True
20,21,1.353522,2.041042,1.720495,-0.078021,1.120634,3.134824,True,-1.528536,True,True,True,True
27,28,0.444588,-1.215229,0.708486,-0.078021,1.815073,2.334698,False,-1.141232,False,False,False,True
29,30,-0.111966,0.644049,-1.174502,-0.078021,-2.314420,2.735547,False,-1.335157,False,False,False,True
40,41,-3.904363,-0.913460,-2.454236,3.823021,1.780134,4.878778,True,-2.953425,True,True,True,True
62,63,-0.791314,-3.504248,-0.358845,-0.078021,-0.773524,3.662635,True,-1.651594,True,True,True,True
73,74,1.436077,-0.064142,1.232859,-0.728194,-2.542225,3.013213,True,-1.454400,False,False,True,True
74,75,-2.053381,-1.113410,1.447048,-0.078021,-1.692052,3.635150,True,-1.485542,False,False,True,True
78,79,0.207806,-0.001118,1.720495,1.222326,-1.067636,3.011649,True,-1.291423,False,False,True,True


**DBSCAN Based Outliers**

In [15]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan.fit(X)

# Get the cluster labels
data['DBSCAN_Labels'] = dbscan.labels_

# Identify outliers (points assigned to the noise cluster, labeled -1)
data['DBSCAN_Outliers'] = data['DBSCAN_Labels'] == -1

data[data['DBSCAN_Outliers']]


,Employee_ID,Monthly_Sales,Customer_Satisfaction,Task_Completion_Rate,Absenteeism_Days,Overtime_Hours,Mahalanobis_Distance,Mahalanobis_Distance_Outliers,LOF_Score,LOF_Outliers,Combined_Outliers,Isolation_Forest_Outliers,OCSVM_Outliers,DBSCAN_Labels,DBSCAN_Outliers
0,1,0.545506,0.303572,-1.047948,-0.728194,0.104498,1.771345,False,-1.028132,False,False,False,False,-1,True
1,2,0.015983,0.518344,-0.281555,0.572153,-0.041920,0.790501,False,-0.975992,False,False,False,False,-1,True
2,3,0.671407,1.070905,0.184125,0.572153,-1.465285,2.092761,False,-1.148195,False,False,False,False,-1,True
3,4,1.401374,1.039959,0.216271,-0.728194,0.666392,1.866704,False,-1.141210,False,False,False,False,-1,True
4,5,-0.063981,-1.532551,-0.166556,-0.728194,0.564400,1.793438,False,-1.021101,False,False,False,False,-1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,0.452609,-0.571359,-0.212730,-0.728194,0.261774,1.118547,False,-0.971931,False,False,False,False,-1,True
196,197,-0.605785,-1.887475,-0.027772,-0.078021,0.921205,2.113868,False,-1.039211,False,False,False,False,-1,True
197,198,-4.058761,1.357436,-4.509134,7.724063,-3.020336,8.487572,True,-4.428641,True,True,True,True,-1,True
198,199,0.179826,-0.196153,-0.162110,-0.728194,2.024806,2.226668,False,-1.143595,False,False,False,False,-1,True
